In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

gold_delivery_performance_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "gold/delivery_performance/"
)

silver_carriers_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/carriers/"
)

gold_late_shipments_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "gold/late_shipments/"
)


# ============================================================
# 2. LECTURE
# ============================================================

delivery_performance = (
    spark.read
    .format("delta")
    .load(gold_delivery_performance_path)
)

silver_carriers = (
    spark.read
    .format("delta")
    .load(silver_carriers_path)
)


# ============================================================
# 3. RÉFÉRENTIEL CARRIERS
# ============================================================

carrier_reference = (
    silver_carriers
    .select(
        "carrier_id",
        "carrier_name",
        "carrier_type",
        "service_level"
    )
)


# ============================================================
# 4. FILTRER LES SHIPMENTS EN RETARD
# ============================================================

late_shipments = (
    delivery_performance

    .filter(
        F.col("is_late") == True
    )

    .join(
        carrier_reference,
        on="carrier_id",
        how="left"
    )
)


# ============================================================
# 5. CONSTRUIRE LA TABLE GOLD
# ============================================================

gold_late_shipments = (
    late_shipments

    .select(
        "shipment_id",
        "order_id",
        "carrier_id",
        "carrier_name",
        "carrier_type",
        "service_level",
        "origin_city",
        "destination_city",
        "shipment_time",
        "expected_delivery_time",
        "delivered_at",
        "shipment_date",
        "delivery_date",
        "delivery_duration_hours",
        "delivery_delay_hours",
        "current_status"
    )

    # Arrondi pour faciliter la lecture analytique
    .withColumn(
        "delivery_duration_hours",
        F.round(
            F.col("delivery_duration_hours"),
            2
        )
    )

    .withColumn(
        "delivery_delay_hours",
        F.round(
            F.col("delivery_delay_hours"),
            2
        )
    )
)


# ============================================================
# 6. ÉCRITURE GOLD
# ============================================================

(
    gold_late_shipments
    .write
    .format("delta")
    .mode("overwrite")
    .save(gold_late_shipments_path)
)

print(
    "gold_late_shipments créée avec succès."
)


# ============================================================
# 7. VÉRIFICATION
# ============================================================

print(
    "Nombre de shipments en retard :",
    gold_late_shipments.count()
)

display(
    gold_late_shipments
    .orderBy(
        F.col("delivery_delay_hours").desc()
    )
)